# Phase 5 — Controlled Model Improvements
Compare two one-pass improvements with the original baseline on the same test set.

In [1]:
import json, random, numpy as np, pandas as pd, torch
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader; from torchvision import models, transforms; from PIL import Image; from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

## Load fixed splits
All models use the same Phase 2 records and untouched test set.

In [2]:
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
root=Path.cwd(); df=pd.read_csv(root/'phase2_clean_splits.csv'); df['target']=df.label.map({'OK':0,'Defective':1})
mean,std=[.485,.456,.406],[.229,.224,.225]
train_tf=transforms.Compose([transforms.Resize((112,112)),transforms.RandomHorizontalFlip(),transforms.RandomRotation(10),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([transforms.Resize((112,112)),transforms.ToTensor(),transforms.Normalize(mean,std)])

## Build deterministic loaders
Random augmentation is restricted to the training loader.

In [3]:
class CastingDS(Dataset):
    def __init__(self,frame,tf): self.frame,self.tf=frame.reset_index(drop=True),tf
    def __len__(self): return len(self.frame)
    def __getitem__(self,i): r=self.frame.iloc[i]; return self.tf(Image.open(r.path).convert('RGB')),int(r.target)
train_dl=DataLoader(CastingDS(df[df.split.eq('train')],train_tf),512,shuffle=True); test_dl=DataLoader(CastingDS(df[df.split.eq('test')],eval_tf),512)
print(len(train_dl.dataset),len(test_dl.dataset))

5098 1093


## Define actual-output metrics
All reported values are calculated from model probabilities and labels.

In [4]:
def metrics(y,p,s):
    pr,rc,f1,_=precision_recall_fscore_support(y,p,average='binary',zero_division=0)
    return {'Accuracy':accuracy_score(y,p),'Precision':pr,'Recall':rc,'F1':f1,'ROC-AUC':roc_auc_score(y,s)}

In [5]:
def predict(m):
    m.eval(); y,p,s=[],[],[]
    with torch.no_grad():
        for x,t in test_dl: o=m(x); y+=t.tolist(); p+=o.argmax(1).tolist(); s+=o.softmax(1)[:,1].tolist()
    return metrics(y,p,s)

## Train low-resolution unweighted head
This tests whether a faster 112×112 input is a useful controlled improvement.

In [6]:
def make():
    m=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT); m.classifier[1]=nn.Linear(m.last_channel,2)
    for p in m.features.parameters(): p.requires_grad=False
    return m
low=make(); opt=torch.optim.Adam(low.classifier.parameters(),1e-3); loss_fn=nn.CrossEntropyLoss()
low.train();
for x,y in train_dl: opt.zero_grad(); loss=loss_fn(low(x),y); loss.backward(); opt.step()

## Train low-resolution class-weighted head
Class weighting targets the observed defective false-negative pattern.

In [7]:
weighted=make(); counts=df[df.split.eq('train')].target.value_counts().sort_index()
weights=torch.tensor(len(df[df.split.eq('train')])/(2*counts.values),dtype=torch.float32); loss_fn=nn.CrossEntropyLoss(weight=weights)
opt=torch.optim.Adam(weighted.classifier.parameters(),1e-3); weighted.train()
for x,y in train_dl: opt.zero_grad(); loss=loss_fn(weighted(x),y); loss.backward(); opt.step()
low_metrics=predict(low); weighted_metrics=predict(weighted); print(low_metrics); print(weighted_metrics)

{'Accuracy': 0.6550777676120768, 'Precision': 0.9381443298969072, 'Recall': 0.4319620253164557, 'F1': 0.5915492957746479, 'ROC-AUC': 0.9175087179768803}
{'Accuracy': 0.4958828911253431, 'Precision': 1.0, 'Recall': 0.1281645569620253, 'F1': 0.22720897615708274, 'ROC-AUC': 0.892353579175705}


## Compare fairly on the fixed unseen test set
The baseline row is read from the original Phase 4 predictions.

In [8]:
base=pd.read_csv(root/'phase4_test_predictions.csv'); baseline=metrics(base.actual,base.predicted,base.defective_score)
comparison=pd.DataFrame([baseline,low_metrics,weighted_metrics]); comparison.insert(0,'Model',['Original MobileNetV2 head','112px unweighted head','112px class-weighted head'])
display(comparison); comparison.to_csv('phase5_model_comparison.csv',index=False)
torch.save(low.state_dict(),'phase5_112px_unweighted_head.pt'); torch.save(weighted.state_dict(),'phase5_112px_class_weighted_head.pt')

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Original MobileNetV2 head,0.940531,0.993043,0.903481,0.946147,0.993503
1,112px unweighted head,0.655078,0.938144,0.431962,0.591549,0.917509
2,112px class-weighted head,0.495883,1.000000,0.128165,0.227209,0.892354


## Select and hand off the measured result
Phase 6 can use the selected model after reviewing the error trade-off.

In [9]:
best=comparison.iloc[comparison.F1.argmax()]; print('Measured best by F1:',best.Model)
with open('phase5_selected_model.json','w') as f: json.dump(best.to_dict(),f,indent=2)
print('Phase 5 complete; continuing to Phase 6.')

Measured best by F1: Original MobileNetV2 head
Phase 5 complete; continuing to Phase 6.
